# Day 37 · Shopify 生态与 API

**配套讲义**: [`days/day-37.md`](../days/day-37.md) ｜ **本地可跑，不需要 GPU**

封装 Admin GraphQL 客户端（分页、重试、限流退避），能从测试店拉到商品列表并落库；并说清三种 App 类型的审核要求差异。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w7.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 自检（不需要真实店铺）

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.shopify.client"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 手写一遍限流退避逻辑

别看源码，自己写一个。核心：**读到低点数就 sleep，而不是等 429**。

In [ ]:
import time

def backoff_sleep(cost_info):
    """点数不足就等，而不是被 429 打回来。"""
    available = cost_info.get("currentlyAvailable", 2000)
    restore_rate = cost_info.get("restoreRate", 50)     # 点/秒
    if available >= 200:
        return 0.0
    wait = (200 - available) / max(restore_rate, 1)
    print(f"  点数 {available} 偏低 → 等 {wait:.2f}s")
    return wait

for ci in ({"currentlyAvailable": 1968, "restoreRate": 50},
           {"currentlyAvailable": 120, "restoreRate": 50},
           {"currentlyAvailable": 12, "restoreRate": 50}):
    print(ci)
    t0 = time.time()
    w = backoff_sleep(ci)
    print(f"  → 需等待 {w:.2f}s（不真的 sleep）")

## 3. 隔离层设计

In [ ]:
import sys; sys.path.insert(0, "..")
from src.shopify.client import to_internal_product

raw = {"id": "gid://shopify/Product/1001", "title": "米白色针织衫",
       "variants": {"edges": [{"node": {"sku": "SKU-1001", "price": "199.00"}}]}}
internal = to_internal_product(raw)
print("外部结构 →", raw)
print("内部结构 →", internal)
print("\n→ 为什么要转换：Shopify 改字段名时，你只需要改这一个函数")

## 验收清单

- [ ] `python -m src.shopify.client` 自检通过
- [ ] 能从测试店拉到 ≥10 个商品并落库（JSON / SQLite 都行）
- [ ] 能说清三种 App 类型的差异和审核要求
- [ ] 知道 `throttleStatus` 怎么读、为什么要退避而不是硬撞

**卡住了？** 回看 [`days/day-37.md`](../days/day-37.md) 第五节「容易踩的坑」。

> **明天**：`days/day-38.md` —— OAuth 安装流程与 App 骨架